# Silver Cleaning (B): istinad və xarici data

**Məsuliyyətli:** Aslan

| Bronze | Silver |
|---|---|
| `bronze.olist_products_dataset` | `silver.products` |
| `bronze.product_category_name_translation` | `silver.category_translation` |
| `bronze.olist_sellers_dataset` | `silver.sellers` |
| `bronze.olist_geolocation_dataset` | `silver.geolocation` |
| `bronze.olist_order_reviews_dataset` | `silver.reviews` |
| `bronze.api_exchange_rates` | `silver.exchange_rates` |
| `bronze.api_weather` | `silver.weather` |





> `silver.reviews` üçün `silver.orders` (A tərəfindən) hazır olmalıdır.

In [2]:
import pandas as pd
from sqlalchemy import text
import sys
from pathlib import Path

# notebook-un olduğu qovluqdan bir yuxarı qovluq (db.py orada)
sys.path.append(str(Path.cwd().parent))

from db import get_engine

engine=get_engine()

In [3]:
"""SELECT sorğusu: nəticəni DataFrame kimi qaytarır."""
def q(sql):
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

def run(sql):
    """CREATE/DROP/INSERT: icra edir və commit edir."""
    with engine.begin() as conn:
        conn.execute(text(sql))

In [5]:
# Təmizləmədən əvvəl bronze datasını yoxlayırıq

q("""
SELECT COLUMN_NAME, DATA_TYPE
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'bronze'
  AND TABLE_NAME = 'olist_products_dataset'
ORDER BY ORDINAL_POSITION
""")



,COLUMN_NAME,DATA_TYPE
0,product_id,varchar
1,product_category_name,varchar
2,product_name_lenght,varchar
3,product_description_lenght,varchar
4,product_photos_qty,varchar
5,product_weight_g,varchar
6,product_length_cm,varchar
7,product_height_cm,varchar
8,product_width_cm,varchar


In [9]:
# Təmizləmədən əvvəl bronze datasını yoxlayırıq

q("""
SELECT COUNT(*) AS setir,                                   -- ümumi sətir sayı
       COUNT(DISTINCT product_id) AS unikal_id,             -- unikal məhsul sayı (dublikat varmı?)
       SUM(CASE WHEN product_category_name = '' THEN 1 ELSE 0 END) AS bos_kateqoriya,  -- boş kateqoriyalar
       SUM(CASE WHEN product_weight_g = '' THEN 1 ELSE 0 END) AS bos_ceki              -- boş çəkilər
FROM bronze.olist_products_dataset
""")

,setir,unikal_id,bos_kateqoriya,bos_ceki
0,32951,32951,610,2


In [ ]:
# Köhnə cədvəl varsa silirik ki, kodu təkrar işlətmək təhlükəsiz olsun
run("DROP TABLE IF EXISTS silver.products")

run(
"""
-- 1-ci mərhələ (c): təmizləmə və tip çevirmələri

WITH c As (
    SELECT
    NULLIF(TRIM(product_id),'') AS product_id,
    LOWER(NULLIF(TRIM(product_category_name),'')) AS category_name,
    -- bronze-da "lenght" yazılış səhvi var, silver-də "length" edirik
    TRY_CAST(NULLIF(TRIM(product_name_length),'') AS INT) AS length_cm,
    TRY_CAST(NULLIF(TRIM(product_description_lenght), '') AS INT) AS description_length,
    TRY_CAST(NULLIF(TRIM(product_photos_qty), '') AS INT) AS photos_qty,
 

)

"""   
)
